In [1]:
import pandas as pd
import numpy as np

In [7]:
# load and check the columns
import pandas as pd

meta = pd.read_csv("meta.csv", encoding="utf-8")
print(meta.columns)
meta.head(15)

Index(['url', 'title', 'studio', 'rating', 'runtime', 'cast', 'director',
       'genre', 'summary', 'awards', 'metascore', 'userscore', 'RelDate'],
      dtype='str')


,url,title,studio,rating,runtime,cast,director,genre,summary,awards,metascore,userscore,RelDate
0,https://www.metacritic.com/movie/!women-art-re...,!Women Art Revolution,Hotwire Productions,| Not Rated,83.0,NaN,Lynn Hershman-Leeson,Documentary,NaN,NaN,70,NaN,2011-06-01
1,https://www.metacritic.com/movie/10-cloverfiel...,10 Cloverfield Lane,Paramount Pictures,| PG-13,104.0,"John Gallagher Jr.,John Goodman,Mary Elizabeth...",Dan Trachtenberg,"Action,Sci-Fi,Drama,Mystery,Thriller,Horror","Waking up from a car accident, a young woman (...","#18MostDiscussedMovieof2016 , #1MostSharedMovi...",76,7.7,2016-03-11
2,https://www.metacritic.com/movie/10-items-or-less,10 Items or Less,Click Star,| R,82.0,"Jonah Hill,Morgan Freeman,Paz Vega",Brad Silberling,"Drama,Comedy,Romance",While researching a role as a supermarket mana...,NaN,54,5.8,2006-12-01
3,https://www.metacritic.com/movie/10-years,10 Years,Anchor Bay Entertainment,| R,100.0,"Channing Tatum,Chris Pratt,Jenna Dewan",Jamie Linden,"Drama,Comedy,Romance",NaN,NaN,61,6.9,2012-09-14
4,https://www.metacritic.com/movie/100-bloody-acres,100 Bloody Acres,Music Box Films,| Not Rated,91.0,NaN,Cameron Cairnes,"Horror,Comedy",Reg and Lindsay run an organic fertilizer busi...,NaN,63,7.5,2013-06-28
5,https://www.metacritic.com/movie/100-streets,100 Streets,Samuel Goldwyn Films,NaN,93.0,NaN,Jim O'Hanlon,Drama,NaN,NaN,44,6.1,2017-01-13
6,https://www.metacritic.com/movie/1000-times-go...,"1,000 Times Good Night",Film Movement,| Not Rated,117.0,NaN,Erik Poppe,Drama,NaN,NaN,57,6.8,2014-10-24
7,https://www.metacritic.com/movie/10000-bc,"10,000 BC",Warner Bros. Pictures,| PG-13,109.0,"Camilla Belle,Marco Khan,Steven Strait",Roland Emmerich,"Adventure,Drama,Fantasy",NaN,"#23MostDiscussedMovieof2008 , #27MostSharedMov...",34,4.6,2008-03-07
8,https://www.metacritic.com/movie/10000-km,"10,000 km",Broad Green Pictures,| R,99.0,NaN,Carlos Marques-Marcet,"Drama,Comedy,Romance","Two people in love, two apartments - one in Ba...",NaN,75,7.4,2015-07-10
9,https://www.metacritic.com/movie/1001-grams,1001 Grams,Kino Lorber,| Not Rated,93.0,NaN,Bent Hamer,Drama,When Norwegian scientist Marie attends a semin...,NaN,65,NaN,2015-05-08


In [8]:
# fix garbled names where needed
def fix_encoding(s):
    try:
        return s.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s

meta['cast'] = meta['cast'].dropna().apply(fix_encoding)

In [11]:
# split into one actor per row and clean
cast_long = meta['cast'].dropna().str.split(',').explode().str.strip()

# drop empty strings (trailing commas) and leftover suffix fragments like "Jr."
cast_long = cast_long[~cast_long.isin(['', 'Jr.', 'Sr.', 'II', 'III'])]
cast_long.head(5)


1         John Gallagher Jr.
1               John Goodman
1    Mary Elizabeth Winstead
2                 Jonah Hill
2             Morgan Freeman
Name: cast, dtype: str

In [20]:
# build table, sort A-Z, assign IDs
cast_member = (
    cast_long
    .replace({'Zsolt PÃ': 'Zsolt Páll'})
    .drop_duplicates()
    .sort_values(key=lambda s: s.str.lower())
    .to_frame(name='castname')
    .reset_index(drop=True)
)
cast_member.insert(0, 'castid', cast_member.index + 1)
cast_member.head(5)

,castid,castname
0,1,50 Cent
1,2,6ix9ine
2,3,A Martinez
3,4,A. Brian Daniels
4,5,A. Patrice Moreau


In [23]:
# check and export
print(cast_member.shape)
print(cast_member[cast_member['castname'].str.contains('Ã|Â', na=False)])  # should be empty
cast_member.head(20)

cast_member.to_csv("cast_member_extract.csv", index=False)

(24861, 2)
Empty DataFrame
Columns: [castid, castname]
Index: []


In [25]:
# check for more weird symbols
odd = sorted({c for name in cast_member['castname'] for c in name if ord(c) > 127})
print(odd)

['À', 'Á', 'Å', 'Æ', 'Ç', 'É', 'Í', 'Ó', 'Ô', 'Ö', 'Ø', 'Ú', 'Ü', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ò', 'ó', 'ô', 'õ', 'ö', 'ø', 'ú', 'û', 'ü', 'ý', 'þ', 'ň', 'ź', 'Ž']


In [26]:
# no other weird characters, let's find Zsolt 
hits = meta[meta['cast'].str.contains('Zsolt P', na=False)][['title', 'url', 'cast']]
print(len(hits))
hits

1


,title,url,cast
4212,In Fabric,https://www.metacritic.com/movie/in-fabric,"Barry Adamson,Caroline Catz,Deborah Griffin,Fa..."


Zsolt PÃ was found, and fixed with .replace({'Zsolt PÃ': 'Zsolt Páll'}) in the code of "# build table, sort A-Z, assign IDs" so the IDs are regenerated with the correct name in its correct alphabetical spot